# ODI to Databricks Migration

**Source File:** TARGET.sql
**Conversion Timestamp:** 2024-07-30 12:00:00 UTC
**Description:** This notebook performs a full load insert into the `trg_dep` table from the `departments` table.

In [ ]:
# SCEN_TASK_NO in {10}: Initialize ETL parameters
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

In [ ]:
display(spark.sql("""
SELECT
  '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
  '${ETL_PROC_WID}' AS ETL_PROC_WID,
  '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

## Insert into Target Table

In [ ]:
%sql
-- SCEN_TASK_NO in {20} & {30}: Insert data into the target department table
INSERT INTO workspace.hr.trg_dep
(
  DEPARTMENT_ID ,
  DEPARTMENT_NAME ,
  MANAGER_ID ,
  LOCATION_ID
)
SELECT
  DEPARTMENTS.DEPARTMENT_ID ,
  DEPARTMENTS.DEPARTMENT_NAME ,
  DEPARTMENTS.MANAGER_ID ,
  DEPARTMENTS.LOCATION_ID
FROM
  workspace.hr.departments AS DEPARTMENTS;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM workspace.hr.trg_dep;

## Conversion Notes

### Manual Actions Required:

1.  **Target Table DDL:** Ensure `workspace.hr.trg_dep` and `workspace.hr.departments` tables exist with appropriate schemas in Databricks Delta Lake.
    *   Example DDL for `workspace.hr.trg_dep` (if not already existing):
        ```sql
        CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep (
            DEPARTMENT_ID BIGINT,
            DEPARTMENT_NAME STRING,
            MANAGER_ID BIGINT,
            LOCATION_ID BIGINT
        ) USING DELTA;
        ```
    *   Example DDL for `workspace.hr.departments` (if not already existing):
        ```sql
        CREATE TABLE IF NOT EXISTS workspace.hr.departments (
            DEPARTMENT_ID BIGINT,
            DEPARTMENT_NAME STRING,
            MANAGER_ID BIGINT,
            LOCATION_ID BIGINT
        ) USING DELTA;
        ```

2.  **Schema Configuration:** The schema `hr` has been automatically converted to `workspace.hr`. Verify this matches your Databricks environment's schema structure.

### Automatic Conversions Applied:

1.  **Oracle Hints Removed:** `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable to Databricks Delta Lake.
2.  **Schema and Table Naming:** All schema and table names (`HR.TRG_DEP`, `HR.DEPARTMENTS`) have been converted to `workspace.hr.trg_dep` and `workspace.hr.departments` respectively, following the `workspace.<schema_lower>.<table_lower>` convention.
3.  **SCEN_TASK_NO Mapping:** Empty `SCEN_TASK_NO` blocks ({10}, {20}) are either implicitly covered by setup cells (like widget initialization) or noted as comments in adjacent code cells. The `INSERT` statement corresponds to `SCEN_TASK_NO {30}`.